In [ ]:
import pandas as pd
import numpy as np

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv(
    "../data/processed/bank_marketing_features.csv"
)

df.shape

In [ ]:
previous = pd.crosstab(
    df["has_previous_contact"],
    df["y"]
)

previous

In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(
    previous
)

print(f"Chi-square: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Statistically significant association.")
else:
    print("No statistically significant association.")

In [ ]:
n = previous.values.sum()

cramers_v = np.sqrt(
    chi2 /
    (n * (min(previous.shape) - 1))
)

print(f"Cramér's V: {cramers_v:.4f}")

In [ ]:
conversion_by_history = (
    df.groupby("has_previous_contact")["y"]
      .agg(
          customers="count",
          conversions="sum",
          conversion_rate="mean"
      )
)

conversion_by_history["conversion_rate"] *= 100

conversion_by_history

In [ ]:
yes_balance = df.loc[
    df["y"] == 1,
    "balance"
]

no_balance = df.loc[
    df["y"] == 0,
    "balance"
]

In [ ]:
print("Subscribed median:", yes_balance.median())
print("Not subscribed median:", no_balance.median())

In [ ]:
stat, p_value = stats.mannwhitneyu(
    yes_balance,
    no_balance,
    alternative="two-sided"
)

print(f"U statistic: {stat:.4f}")
print(f"p-value: {p_value:.6f}")

In [ ]:
contact_table = pd.crosstab(
    df["contact"],
    df["y"]
)

contact_table

In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(
    contact_table
)

print(f"Chi-square: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")

In [ ]:
df["campaign_group"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 2, 3, 5, 10, np.inf],
    labels=[
        "1",
        "2",
        "3",
        "4-5",
        "6-10",
        "11+"
    ]
)

In [ ]:
campaign_analysis = (
    df.groupby(
        "campaign_group",
        observed=True
    )["y"]
    .agg(
        customers="count",
        conversions="sum",
        conversion_rate="mean"
    )
)

campaign_analysis["conversion_rate"] *= 100

campaign_analysis

In [ ]:
model_df = df[
    [
        "y",
        "age",
        "balance",
        "campaign",
        "previous",
        "has_previous_contact"
    ]
].dropna()

In [ ]:
X = model_df[
    [
        "age",
        "balance",
        "campaign",
        "previous",
        "has_previous_contact"
    ]
]

X = sm.add_constant(X)

y = model_df["y"]

logit_model = sm.Logit(y, X).fit()

print(logit_model.summary())

In [ ]:
odds_ratios = pd.DataFrame({
    "feature": logit_model.params.index,
    "odds_ratio": np.exp(
        logit_model.params.values
    ),
    "p_value": logit_model.pvalues.values
})

odds_ratios

In [ ]:
results = []

tests = [
    (
        "Previous contact vs conversion",
        "Chi-square"
    ),
    (
        "Contact method vs conversion",
        "Chi-square"
    ),
    (
        "Balance vs conversion",
        "Mann-Whitney U"
    )
]

for name, test in tests:
    results.append({
        "Question": name,
        "Test": test
    })

pd.DataFrame(results)